# Night Flash — Google Colab

Seven cells. Run them top to bottom; only cell 3 needs editing.

| cell | what it does |
|---|---|
| 1 | install and import |
| 2 | mount Drive, point at the data |
| 3 | **configuration — one `Session` object** |
| 4 | `train` — read the briefing it prints |
| 5 | inference and previews |
| 6 | where the run lives, and how to move it to Kaggle |

The TUI does not run in Colab. `nightflash-tui` is for your Mac.

Kaggle instructions, the command cookbook and the reasoning behind each
setting: **`docs/QuickStartKaggle.md`**. Colab-specific notes:
**`docs/COLAB.md`**.

### Before you start

1. Put your photo folders in Drive: `normal_photos` (source) and
   `disposable_photos` (target), plus `validation_photos` of held-out
   night-flash photos.
2. Accelerator: **GPU T4**. Colab gives one.
3. Do **not** use a TPU — see `docs/QuickStartKaggle.md` for why.

Check the target domain before you spend GPU hours on it — see
`docs/QuickStartKaggle.md` § *Before you start*.

### Persistence

Drive survives a disconnect, so `runs_root` points there and there is nothing
to snapshot: leave `snapshot_every` at 0. `s.auto_resume()` reports that this
is not a Kaggle runtime and does nothing, so the cell is still safe to run.

Colab disconnects idle sessions after about 90 minutes, so a shorter
`extra_epochs` per run gives you more save points.

### Inference notes

Whole-frame is the default and preserves the exact original size. For
native-resolution detail use `s.infer(photo, tile=256)` — and keep `tile` equal
to `image_size`. Add `max_edge=1024` to downscale while keeping aspect ratio.

In [ ]:
# ── 1. install ──────────────────────────────────────────────────────
import subprocess, sys, shutil, os
from pathlib import Path

REPO_URL = ""      # optional: set to a git URL, otherwise upload the zip
PROJECT = Path("/content/nightflash-edge")

if REPO_URL:
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
else:
    if not (PROJECT / "pyproject.toml").exists():
        zips = sorted(Path("/content").glob("*.zip"))
        if not zips:
            raise RuntimeError(
                "Upload dist/nightflash-edge-colab.zip to /content/ via the file "
                "browser, or set REPO_URL at the top of this cell."
            )
        subprocess.run(["unzip", "-q", str(zips[0]), "-d", "/content"], check=True)

assert (PROJECT / "pyproject.toml").exists()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / "src"))

import torch
from nightflash import Session
from nightflash.briefing import brief
from nightflash.notebook import show_previews, show_image
from nightflash.kaggle import environment_report

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu   ", torch.cuda.get_device_name(0))

In [ ]:
# ── 2. data ────────────────────────────────────────────────────────
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/nightflash-edge")  # edit if different
DATASET = DRIVE_ROOT / "training_photos"
RUNS = DRIVE_ROOT / "runs"

SOURCE_DIR = DATASET / "normal_photos"
TARGET_DIR = DATASET / "disposable_photos"
VAL_DIR = DATASET / "validation_photos"
MASK_DIR = None  # set to DATASET/"source_masks" when you have Vision masks

for label, path in [("source", SOURCE_DIR), ("target", TARGET_DIR)]:
    if not path.is_dir():
        raise FileNotFoundError(f"{label} not found: {path}")
    n = sum(1 for p in path.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    print(f"{label:7s} {n:6d} images")
print("runs root:", RUNS)

In [ ]:
# ── 3. configuration — the only thing you normally edit ─────────────
s = Session(
    arch="edgehat",          # edgehat | swinir | restormer | hybrid
    head="freqsplit",        # tanh | perpixel | freqsplit | lut3d
                             #   | lut3d+freqsplit | lut3d+perpixel | tanh+freqsplit
    name="night",            # run folder suffix: edgehat_freqsplit_<name>
    source_dir=SOURCE_DIR,
    target_dir=TARGET_DIR,
    val_dir=VAL_DIR,
    mask_dir=MASK_DIR,
    runs_root=RUNS,
    image_size=256,          # keep --tile equal to this for tiled inference
    batch_size=8,
    epochs=50,               # ABSOLUTE target epoch, not a count of extra epochs
    device="auto",           # "auto" resolves to cuda on Kaggle/Colab, mps on Mac
    use_amp=True,
    num_workers=2,
    preview_every=250,
    log_every=25,
    halation=True,
    subject_gate=True,
    region_exposure=True,
    augment_preset="default",   # default | conservative | strong | none

)
print(s)

print(brief(s))  # full pre-flight plan

In [ ]:
# ── 4. resume, then train ─────────────────────────────────────────
# On Colab, runs_root is inside Drive, so there is nothing to restore.
# s.auto_resume() reports that it is not a Kaggle runtime and does nothing.
# Pull in the last snapshot automatically, or start fresh.
# Safe to run every session, and a no-op outside Kaggle.
s.auto_resume()

# Read this before training. It prints the plan, the flags, the paths, the
# flow, the current state, and a numbered decision menu.
print(brief(s, mode="train"))

# Pick ONE. Each line is idempotent and safe to re-run.
s.train(extra_epochs=20)        # relative: 20 more from wherever the run is
# s.train()                    # absolute: go to Session.epochs (cell 3)

In [ ]:
# ── 5. look at the output ─────────────────────────────────────────
# Pick ONE. Each line is idempotent and safe to re-run.

s.infer(f"{SOURCE_DIR}", seeds=2, save_comparison=True)   # whole frame, 1:1
# s.infer(photo, tile=256)                  # native detail; tile == image_size
# s.infer(photo, tile=256, tile_overlap=64) # faster, slightly more seams
# s.infer(photo, max_edge=1024)             # downscale long edge, keep ratio
# s.infer(photo, max_edge=1024, tile=256)  # downscale AND tile
# s.infer(photo, seeds=3)                   # 3 stochastic variants

show_previews(s.run_dir, n=4)
print("\nRun directory:", s.run_dir)
print("State:", s.state().status)
print("Inference output:", s.inference_dir)

In [ ]:
# ── 6. keep the run ──────────────────────────────────────────────
# Colab Drive persists, so there is nothing to snapshot. The run lives at:
print("Run directory:", s.run_dir)
print("State:", s.state().status)
print("Checkpoint:", s.latest if s.latest.exists() else "(none yet)")

# Save the notebook's output, or download the run directory, before the
# session times out. Colab disconnects idle sessions after ~90 minutes.
#
# To move a finished run to Kaggle, zip the run directory and upload it as a
# Dataset; s.auto_resume() there will pick it up.